# Parameter-Efficient Fine-Tuning (PEFT): Adapting Large Language Models Efficiently

In the previous notebook, we saw how **quantization** makes large language models smaller and cheaper to run during inference.

That naturally leads to the next engineering question:

> How can we adapt a large language model to a new task without retraining billions of parameters?

This notebook explains **Parameter-Efficient Fine-Tuning (PEFT)** with a practical focus on LoRA and QLoRA: the intuition, the math, the memory trade-offs, and the kind of workflow used in real LLM systems.


## Notebook Roadmap

We will build from the cost problem toward a practical adaptation workflow:

- Why full fine-tuning is expensive
- What PEFT changes
- How LoRA works internally
- Why QLoRA combines quantization and LoRA
- How to structure a small Hugging Face PEFT workflow
- How PEFT affects deployment, storage, and experimentation


In [ ]:
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.2,
    "font.size": 11,
})

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## Why Full Fine-Tuning Is Expensive

Training a modern LLM from scratch is extremely expensive, but even **traditional fine-tuning** can be costly because it updates every parameter.

When a model has billions of parameters, full fine-tuning usually means:

- every weight can receive gradients
- optimizer states must be stored for every trainable weight
- checkpoints are almost as large as the base model
- each experiment creates another huge model artifact

For a single 7B model, this can quickly become a GPU memory and storage problem rather than a modeling problem.


In [ ]:
def estimate_full_finetuning_cost(parameters, dtype_bytes=2, optimizer_multiplier=2):
    """Estimate rough memory and checkpoint size for full fine-tuning.

    Assumptions:
    - model weights are stored in fp16/bf16 by default
    - gradients use one extra copy
    - Adam-like optimizer states use two extra copies
    """
    weight_gb = parameters * dtype_bytes / 1e9
    gradient_gb = weight_gb
    optimizer_gb = weight_gb * optimizer_multiplier
    training_state_gb = weight_gb + gradient_gb + optimizer_gb
    checkpoint_gb = weight_gb
    return weight_gb, optimizer_gb, training_state_gb, checkpoint_gb


model_sizes = {
    "100M": 100_000_000,
    "1B": 1_000_000_000,
    "7B": 7_000_000_000,
    "13B": 13_000_000_000,
    "70B": 70_000_000_000,
}

rows = []
for name, params in model_sizes.items():
    weight_gb, optimizer_gb, training_state_gb, checkpoint_gb = estimate_full_finetuning_cost(params)
    rows.append({
        "Model": name,
        "Total Parameters": f"{params:,}",
        "Trainable Parameters": f"{params:,}",
        "Weights (GB, fp16)": weight_gb,
        "Optimizer State (GB)": optimizer_gb,
        "Approx. Training State (GB)": training_state_gb,
        "Checkpoint Size (GB)": checkpoint_gb,
    })

full_ft_cost = pd.DataFrame(rows)
full_ft_cost


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
labels = list(model_sizes.keys())
training_memory = [estimate_full_finetuning_cost(p)[2] for p in model_sizes.values()]
checkpoint_size = [estimate_full_finetuning_cost(p)[3] for p in model_sizes.values()]

x = np.arange(len(labels))
width = 0.36
ax.bar(x - width / 2, training_memory, width, label="Approx. training state")
ax.bar(x + width / 2, checkpoint_size, width, label="One fp16 checkpoint")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("GB")
ax.set_title("Full Fine-Tuning Scales Quickly")
ax.legend()
plt.show()


The exact memory footprint depends on framework, optimizer, activation checkpointing, sequence length, batch size, and distributed strategy. The key pattern is stable: **full fine-tuning scales with the full model size**.

PEFT attacks that scaling problem by making only a tiny fraction of parameters trainable.


## What Is PEFT?

**Parameter-Efficient Fine-Tuning** keeps almost all pretrained model weights frozen and trains only a small number of additional or selected parameters.

Instead of asking:

> How do we update this entire model?

PEFT asks:

> What is the smallest useful set of parameters we can train to steer this model toward a task?

That shift dramatically reduces memory use, training time, checkpoint size, and the cost of running many experiments.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.axis("off")

def labeled_box(ax, xy, width, height, label, color):
    patch = Rectangle(xy, width, height, facecolor=color, edgecolor="#243042", linewidth=1.5)
    ax.add_patch(patch)
    ax.text(xy[0] + width / 2, xy[1] + height / 2, label, ha="center", va="center", weight="bold")

labeled_box(ax, (0.05, 0.55), 0.35, 0.25, "Full Fine-Tuning\nUpdate all weights", "#f7c6b8")
labeled_box(ax, (0.60, 0.55), 0.35, 0.25, "PEFT\nFreeze base + train small adapter", "#b9e4c9")

for i in range(6):
    ax.add_patch(Rectangle((0.08 + i * 0.045, 0.18), 0.035, 0.22, facecolor="#d95f59", edgecolor="white"))
    ax.add_patch(Rectangle((0.63 + i * 0.045, 0.18), 0.035, 0.22, facecolor="#9aa6b2", edgecolor="white"))

ax.add_patch(Rectangle((0.88, 0.18), 0.035, 0.22, facecolor="#2f9e44", edgecolor="white"))
ax.text(0.20, 0.08, "Many trainable blocks", ha="center")
ax.text(0.75, 0.08, "Mostly frozen blocks", ha="center")
ax.text(0.90, 0.08, "Small trainable adapter", ha="center")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.show()


## Different PEFT Methods

PEFT is a family of methods. They share the same engineering idea, but they inject trainable capacity in different places.


In [ ]:
peft_methods = pd.DataFrame([
    {
        "Method": "Adapters",
        "Idea": "Insert small neural modules between transformer layers.",
        "Memory": "Low",
        "Training Speed": "Fast",
        "Popularity": "Medium",
        "Inference Cost": "Can add extra layers",
    },
    {
        "Method": "Prefix Tuning",
        "Idea": "Learn virtual key/value prefixes for attention.",
        "Memory": "Very low",
        "Training Speed": "Fast",
        "Popularity": "Medium",
        "Inference Cost": "Longer effective context",
    },
    {
        "Method": "Prompt Tuning",
        "Idea": "Learn soft prompt embeddings prepended to input.",
        "Memory": "Very low",
        "Training Speed": "Fast",
        "Popularity": "Medium",
        "Inference Cost": "Longer effective prompt",
    },
    {
        "Method": "LoRA",
        "Idea": "Learn low-rank updates for selected weight matrices.",
        "Memory": "Low",
        "Training Speed": "Fast",
        "Popularity": "Very high",
        "Inference Cost": "Can be merged",
    },
    {
        "Method": "QLoRA",
        "Idea": "Train LoRA adapters while loading the base model in 4-bit.",
        "Memory": "Very low",
        "Training Speed": "Fast",
        "Popularity": "Very high",
        "Inference Cost": "Depends on merge/quantized serving",
    },
])

peft_methods


## Understanding LoRA

LoRA stands for **Low-Rank Adaptation**.

In a transformer, many important operations are linear projections. A simplified linear layer computes:

\[
y = xW
\]

Full fine-tuning updates the original weight matrix \(W\). LoRA freezes \(W\) and learns a small update:

\[
y = x(W + \Delta W)
\]

Instead of learning a full \(\Delta W\), LoRA represents it as two smaller matrices:

\[
\Delta W = A B
\]

If \(W\) is large but the rank \(r\) is small, \(A\) and \(B\) contain far fewer trainable parameters than \(W\).

The next diagrams visualize how that low-rank update sits beside the frozen linear layer.


### Why Low Rank Helps

Suppose a linear layer maps from 4,096 input features to 4,096 output features.

Full fine-tuning would train:

\[
4096 \times 4096 = 16,777,216
\]

parameters for that one matrix.

With LoRA rank \(r = 8\), the trainable parameters are:

\[
4096 \times 8 + 8 \times 4096 = 65,536
\]

That is less than half a percent of the original matrix.


In [ ]:
def lora_parameter_count(in_features, out_features, rank):
    """Return full matrix parameters and LoRA adapter parameters."""
    full = in_features * out_features
    lora = in_features * rank + rank * out_features
    return full, lora


full, lora = lora_parameter_count(4096, 4096, rank=8)
print(f"Full matrix parameters: {full:,}")
print(f"LoRA trainable parameters: {lora:,}")
print(f"Percentage updated: {100 * lora / full:.3f}%")


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.axis("off")

def box(x, y, w, h, text, color):
    ax.add_patch(Rectangle((x, y), w, h, facecolor=color, edgecolor="#1f2937", linewidth=1.4))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", weight="bold")

box(0.04, 0.45, 0.18, 0.20, "Input\nx", "#dbeafe")
box(0.31, 0.45, 0.20, 0.20, "Frozen Weight\nW", "#e5e7eb")
box(0.61, 0.62, 0.12, 0.16, "A", "#bbf7d0")
box(0.76, 0.62, 0.12, 0.16, "B", "#bbf7d0")
box(0.67, 0.30, 0.18, 0.16, "Low-rank update\nDelta W = AB", "#86efac")
box(0.43, 0.08, 0.25, 0.16, "Output\ny = x(W + AB)", "#fde68a")

arrows = [
    ((0.22, 0.55), (0.31, 0.55)),
    ((0.51, 0.55), (0.60, 0.38)),
    ((0.73, 0.70), (0.76, 0.70)),
    ((0.76, 0.62), (0.76, 0.46)),
    ((0.68, 0.30), (0.58, 0.24)),
]
for start, end in arrows:
    ax.add_patch(FancyArrowPatch(start, end, arrowstyle="->", mutation_scale=15, linewidth=1.6))

ax.text(0.41, 0.70, "Base model stays frozen", ha="center", color="#374151")
ax.text(0.75, 0.84, "Only these matrices train", ha="center", color="#166534")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.show()


In [ ]:
matrix_shapes = {
    "W": (16, 16),
    "A": (16, 3),
    "B": (3, 16),
    "Delta W": (16, 16),
}

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, (name, shape) in zip(axes, matrix_shapes.items()):
    data = np.random.normal(size=shape)
    ax.imshow(data, cmap="viridis", aspect="auto")
    ax.set_title(f"{name}\n{shape[0]} x {shape[1]}")
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle("Low-Rank Matrices Reconstruct a Full-Size Update", y=1.08)
plt.tight_layout()
plt.show()


## Tiny LoRA Simulation

The next cells use **NumPy only**. We will create a tiny frozen weight matrix, a low-rank pair of LoRA matrices, and the reconstructed update.


In [ ]:
def make_lora_update(in_features, out_features, rank, scale=0.1):
    """Create a tiny LoRA update using random low-rank matrices."""
    a = np.random.normal(0, scale, size=(in_features, rank))
    b = np.random.normal(0, scale, size=(rank, out_features))
    delta_w = a @ b
    return a, b, delta_w


W = np.array([
    [0.20, -0.10, 0.40, 0.70],
    [0.50, 0.30, -0.20, 0.10],
    [-0.30, 0.80, 0.60, -0.50],
    [0.90, -0.40, 0.20, 0.30],
])

A, B, delta_W = make_lora_update(in_features=4, out_features=4, rank=2)
W_adapted = W + delta_W

print("Frozen W:")
print(np.round(W, 3))
print("\nLoRA A:")
print(np.round(A, 3))
print("\nLoRA B:")
print(np.round(B, 3))
print("\nDelta W = A @ B:")
print(np.round(delta_W, 3))


In [ ]:
x = np.array([[1.0, 0.5, -0.2, 0.1]])

base_output = x @ W
adapted_output = x @ W_adapted

print("Base output:   ", np.round(base_output, 4))
print("Adapted output:", np.round(adapted_output, 4))
print("Change:        ", np.round(adapted_output - base_output, 4))


In [ ]:
full_params = W.size
lora_params = A.size + B.size

pd.DataFrame([
    {"Approach": "Full update", "Trainable Parameters": full_params, "Percentage of W": 100.0},
    {"Approach": "LoRA rank=2", "Trainable Parameters": lora_params, "Percentage of W": 100 * lora_params / full_params},
])


In this tiny 4 x 4 example, rank 2 does not save parameters because the matrix is too small. LoRA becomes powerful when the real model matrices are thousands of dimensions wide.


## Comparing Trainable Parameters


In [ ]:
def estimate_lora_params(total_parameters, target_fraction=0.35, rank=8, hidden_size=4096):
    """Estimate LoRA trainable parameters for a transformer-like model.

    This simplified estimate assumes LoRA is applied to a fraction of linear
    projection parameters and uses a hidden-size-based low-rank formula.
    """
    target_parameters = total_parameters * target_fraction
    matrix_count = max(1, int(target_parameters / (hidden_size * hidden_size)))
    trainable_per_matrix = 2 * hidden_size * rank
    return matrix_count * trainable_per_matrix


def compare_training_strategies(model_sizes, rank=8):
    """Compare full fine-tuning, LoRA, and QLoRA-style adaptation."""
    rows = []
    for model_name, params in model_sizes.items():
        lora_params = estimate_lora_params(params, rank=rank)
        for strategy, trainable, base_bits in [
            ("Full fine-tuning", params, 16),
            ("LoRA", lora_params, 16),
            ("QLoRA", lora_params, 4),
        ]:
            base_memory_gb = params * base_bits / 8 / 1e9
            trainable_memory_gb = trainable * 2 / 1e9
            rows.append({
                "Model": model_name,
                "Strategy": strategy,
                "Total Parameters": params,
                "Trainable Parameters": int(trainable),
                "Percentage Updated": 100 * trainable / params,
                "Approx. Base Memory (GB)": base_memory_gb,
                "Approx. Trainable Weight Memory (GB)": trainable_memory_gb,
            })
    return pd.DataFrame(rows)


strategy_df = compare_training_strategies(model_sizes, rank=8)
strategy_df


In [ ]:
display_df = strategy_df.copy()
display_df["Total Parameters"] = display_df["Total Parameters"].map(lambda x: f"{x:,}")
display_df["Trainable Parameters"] = display_df["Trainable Parameters"].map(lambda x: f"{x:,}")
display_df["Percentage Updated"] = display_df["Percentage Updated"].map(lambda x: f"{x:.3f}%")
display_df["Approx. Base Memory (GB)"] = display_df["Approx. Base Memory (GB)"].map(lambda x: f"{x:.2f}")
display_df["Approx. Trainable Weight Memory (GB)"] = display_df["Approx. Trainable Weight Memory (GB)"].map(lambda x: f"{x:.3f}")
display_df


In [ ]:
pivot = strategy_df.pivot(index="Model", columns="Strategy", values="Percentage Updated").loc[labels]

fig, ax = plt.subplots(figsize=(10, 5))
pivot[["Full fine-tuning", "LoRA", "QLoRA"]].plot(kind="bar", ax=ax)
ax.set_ylabel("Percentage of parameters updated")
ax.set_title("Trainable Parameter Share")
ax.set_yscale("log")
ax.legend(title="")
plt.xticks(rotation=0)
plt.show()


In [ ]:
memory_pivot = strategy_df.pivot(index="Model", columns="Strategy", values="Approx. Base Memory (GB)").loc[labels]

fig, ax = plt.subplots(figsize=(10, 5))
memory_pivot[["Full fine-tuning", "LoRA", "QLoRA"]].plot(kind="bar", ax=ax)
ax.set_ylabel("Approx. base model memory (GB)")
ax.set_title("Quantized Loading Reduces Base Model Memory")
ax.legend(title="")
plt.xticks(rotation=0)
plt.show()


## QLoRA

QLoRA combines two ideas from this learning path:

1. **Quantization** loads the base model in a smaller numerical format.
2. **LoRA** trains small low-rank adapters while the base model remains frozen.

In practice, QLoRA became popular because it made fine-tuning larger models possible on much smaller hardware than full fine-tuning would require.


### QLoRA Building Blocks

The core QLoRA ingredients are:

- **4-bit quantization**: stores frozen base model weights using only 4 bits per value.
- **NF4**: a 4-bit data type designed for normally distributed neural network weights.
- **Double quantization**: quantizes quantization constants to save additional memory.
- **Paged optimizers**: reduce memory spikes during optimization by managing optimizer state more carefully.

The intuition is simple: keep the huge base model compressed, and spend training memory only on the small adapter parameters.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.axis("off")

stages = [
    ("Pretrained\nbase model", "#dbeafe"),
    ("4-bit\nquantized base", "#bfdbfe"),
    ("LoRA\nadapters", "#bbf7d0"),
    ("Fine-tuned\nbehavior", "#fde68a"),
]

for i, (label, color) in enumerate(stages):
    x0 = 0.06 + i * 0.24
    ax.add_patch(Rectangle((x0, 0.38), 0.16, 0.24, facecolor=color, edgecolor="#1f2937", linewidth=1.3))
    ax.text(x0 + 0.08, 0.50, label, ha="center", va="center", weight="bold")
    if i < len(stages) - 1:
        ax.add_patch(FancyArrowPatch((x0 + 0.17, 0.50), (x0 + 0.23, 0.50), arrowstyle="->", mutation_scale=15))

ax.text(0.50, 0.20, "QLoRA = quantized frozen base + trainable LoRA adapters", ha="center", fontsize=12)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.show()


## PEFT with Hugging Face

The common Python ecosystem for PEFT workflows includes:

- `transformers` for model and tokenizer loading
- `peft` for LoRA and adapter configuration
- `datasets` for training data
- `trl` for supervised fine-tuning and preference optimization workflows

The following cell is intentionally safe: it checks whether optional libraries are installed and prints installation guidance if they are not.


In [ ]:
def check_optional_packages(package_names):
    """Report whether optional packages are available."""
    import importlib.util

    availability = {}
    for package in package_names:
        availability[package] = importlib.util.find_spec(package) is not None
    return availability


optional_packages = check_optional_packages(["transformers", "peft", "datasets", "trl"])
pd.DataFrame(
    [{"Package": package, "Available": available} for package, available in optional_packages.items()]
)


In [ ]:
if not all(optional_packages[pkg] for pkg in ["transformers", "peft", "datasets"]):
    print("Optional Hugging Face packages are not all installed.")
    print("To run the live PEFT example, install:")
    print("pip install transformers peft datasets")
else:
    print("Core Hugging Face PEFT packages are available.")


### Educational Hugging Face Workflow

This is the shape of a real PEFT setup. The cell avoids downloading a large model by default. For a small CPU-friendly experiment, use a tiny model such as `sshleifer/tiny-gpt2`.


In [ ]:
HUGGING_FACE_EXAMPLE = r'''
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

model_name = "sshleifer/tiny-gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["c_attn"],
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()
'''

print(HUGGING_FACE_EXAMPLE)


## Mini Project: A Lightweight PEFT Pipeline


In [ ]:
class TinyLinearModel:
    """A tiny stand-in for a language model projection layer."""

    def __init__(self, in_features, out_features):
        self.weight = np.random.normal(0, 0.02, size=(in_features, out_features))
        self.requires_grad = False


class TinyLoRAAdapter:
    """Minimal LoRA adapter for demonstration."""

    def __init__(self, in_features, out_features, rank, alpha=16):
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        self.a = np.random.normal(0, 0.01, size=(in_features, rank))
        self.b = np.zeros((rank, out_features))

    @property
    def trainable_parameters(self):
        return self.a.size + self.b.size

    def update(self):
        return self.scaling * (self.a @ self.b)


def apply_tiny_lora(model, rank=4, alpha=16):
    """Attach a LoRA adapter to a tiny frozen model."""
    in_features, out_features = model.weight.shape
    adapter = TinyLoRAAdapter(in_features, out_features, rank=rank, alpha=alpha)
    return adapter


tiny_model = TinyLinearModel(in_features=32, out_features=32)
tiny_adapter = apply_tiny_lora(tiny_model, rank=4)

print(f"Base weights frozen: {not tiny_model.requires_grad}")
print(f"Base parameters: {tiny_model.weight.size:,}")
print(f"Trainable LoRA parameters: {tiny_adapter.trainable_parameters:,}")
print(f"Percentage updated: {100 * tiny_adapter.trainable_parameters / tiny_model.weight.size:.2f}%")


In [ ]:
def prepare_instruction_examples():
    """Create a tiny instruction-style dataset for demonstration."""
    return pd.DataFrame([
        {
            "instruction": "Summarize the support ticket.",
            "input": "The customer cannot reset their password after changing phones.",
            "target": "Customer needs help with password reset after phone change.",
        },
        {
            "instruction": "Classify the request.",
            "input": "Please update the billing email for our workspace.",
            "target": "Account administration",
        },
        {
            "instruction": "Rewrite professionally.",
            "input": "Your thing is broken and I need it fixed now.",
            "target": "I am experiencing an issue and would appreciate urgent assistance.",
        },
    ])


mini_dataset = prepare_instruction_examples()
mini_dataset


In [ ]:
def format_instruction(row):
    """Format one row as an instruction-tuning prompt."""
    return (
        f"### Instruction:\n{row['instruction']}\n\n"
        f"### Input:\n{row['input']}\n\n"
        f"### Response:\n{row['target']}"
    )


formatted_examples = mini_dataset.apply(format_instruction, axis=1)
print(formatted_examples.iloc[0])


In [ ]:
def peft_project_checklist():
    """Return the practical steps for a PEFT fine-tuning project."""
    return pd.DataFrame([
        {"Step": 1, "Task": "Choose a base model", "Engineering Note": "Match license, latency, context length, and quality needs."},
        {"Step": 2, "Task": "Prepare task data", "Engineering Note": "Use clean instruction/response pairs and evaluation splits."},
        {"Step": 3, "Task": "Quantize if needed", "Engineering Note": "Use 4-bit loading when GPU memory is tight."},
        {"Step": 4, "Task": "Apply LoRA", "Engineering Note": "Select target modules, rank, alpha, and dropout."},
        {"Step": 5, "Task": "Fine-tune lightly", "Engineering Note": "Track validation quality and overfitting."},
        {"Step": 6, "Task": "Save adapter", "Engineering Note": "Adapters are small and easy to version."},
        {"Step": 7, "Task": "Deploy or merge", "Engineering Note": "Serve adapter separately or merge for simpler inference."},
    ])


peft_project_checklist()


## Practical Workflow


In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.axis("off")

workflow = [
    "Choose\nBase Model",
    "Quantize\nOptional",
    "Apply\nLoRA",
    "Fine-Tune",
    "Merge Adapter\nOptional",
    "Deploy",
]

for i, step in enumerate(workflow):
    x0 = 0.03 + i * 0.16
    ax.add_patch(Rectangle((x0, 0.40), 0.12, 0.24, facecolor="#e0f2fe", edgecolor="#0f172a", linewidth=1.2))
    ax.text(x0 + 0.06, 0.52, step, ha="center", va="center", weight="bold")
    if i < len(workflow) - 1:
        ax.add_patch(FancyArrowPatch((x0 + 0.125, 0.52), (x0 + 0.155, 0.52), arrowstyle="->", mutation_scale=14))

ax.set_title("Typical PEFT Workflow", pad=12)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.show()


## Adapter Concept Visualization

One strong deployment advantage of PEFT is that a single base model can support many task-specific adapters.

For example, an organization might keep one approved base model and attach different LoRA adapters for support, legal, medical, code, or internal documentation tasks.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.axis("off")

ax.add_patch(Rectangle((0.38, 0.40), 0.24, 0.22, facecolor="#dbeafe", edgecolor="#111827", linewidth=1.5))
ax.text(0.50, 0.51, "Frozen Base Model", ha="center", va="center", weight="bold")

adapters = [
    ("Support\nAdapter", 0.08, 0.72),
    ("Legal\nAdapter", 0.08, 0.15),
    ("Medical\nAdapter", 0.73, 0.72),
    ("Code\nAdapter", 0.73, 0.15),
]

for label, x0, y0 in adapters:
    ax.add_patch(Rectangle((x0, y0), 0.18, 0.16, facecolor="#bbf7d0", edgecolor="#166534", linewidth=1.2))
    ax.text(x0 + 0.09, y0 + 0.08, label, ha="center", va="center", weight="bold")
    ax.add_patch(FancyArrowPatch((x0 + 0.18 if x0 < 0.5 else x0, y0 + 0.08), (0.38 if x0 < 0.5 else 0.62, 0.51), arrowstyle="->", mutation_scale=14))

ax.set_title("Multiple Small Adapters Can Share One Base Model")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.show()


## Advantages

PEFT is popular because it solves practical engineering problems:

- **Very low memory usage** compared with full fine-tuning
- **Fast training** because fewer parameters receive gradients
- **Small checkpoints** that are easier to store, upload, and version
- **Easy experimentation** across ranks, datasets, and tasks
- **Multiple adapters for one base model**, which simplifies product variants


## Limitations

PEFT is powerful, but it is not magic:

- It still depends heavily on the quality of the base model.
- Small adapters have limited capacity.
- Adapter management can become operationally messy at scale.
- Some tasks may still benefit from full fine-tuning.
- Evaluation remains essential because lower training cost does not guarantee better behavior.


## Real-World Applications

PEFT is widely used when teams want specialization without maintaining a completely separate model for every use case:

- Domain adaptation for company-specific language
- Medical assistants with specialized terminology
- Legal assistants for contract and policy workflows
- Customer support automation
- Enterprise chatbots
- Code assistants
- RAG systems that need task-specific response style or formatting


## Common Misconceptions

**LoRA does not replace the model.** It learns a small update that modifies how the base model behaves.

**Adapters are not standalone models.** They usually need the original base model architecture and weights.

**PEFT is still fine-tuning.** The difference is that only a small number of parameters are trainable.

**Quantization and PEFT solve different problems.** Quantization reduces numerical memory; PEFT reduces the number of trainable parameters. QLoRA combines both.


In [ ]:
summary = pd.DataFrame([
    {"Concept": "Full fine-tuning", "What changes?": "All model weights", "Best when": "Maximum task capacity is needed and compute is available"},
    {"Concept": "LoRA", "What changes?": "Small low-rank adapter matrices", "Best when": "Efficient specialization and experimentation are needed"},
    {"Concept": "QLoRA", "What changes?": "LoRA adapters with quantized frozen base", "Best when": "GPU memory is limited"},
    {"Concept": "Merged adapter", "What changes?": "Adapter update is folded into base weights", "Best when": "Simpler inference deployment is preferred"},
])
summary


## Final Summary

PEFT matters because it makes LLM adaptation practical. Instead of retraining billions of parameters, teams can freeze the base model and train a much smaller set of task-specific parameters.

LoRA is the most common PEFT method because it is simple, effective, and deployment-friendly. It learns a low-rank update \(\Delta W = AB\), which can dramatically reduce trainable parameters while still steering model behavior.

QLoRA became popular because it combines the memory benefits of quantization with the training efficiency of LoRA. This made larger-model fine-tuning accessible on smaller hardware.

In real systems, PEFT supports faster iteration, smaller artifacts, cleaner adapter versioning, and multiple specialized behaviors from one shared base model.


## Next Notebook

➡️ **Next: RLHF & DPO**

After adapting a model to new knowledge or a new task, the next challenge is aligning it with human preferences.

The next notebook will explain how preference optimization methods such as RLHF and DPO help tune models toward responses people actually prefer.
